# 📖 Notebook 1 — The Problem: Stale Replicas Return Old Data

In a replicated key-value store, the *same* key lives on several machines (**replicas**). This gives you fault tolerance and read scalability — but replicas can **drift out of sync**. Reasons include:

1. A write reached replicas `r1` and `r2` but the packet to `r3` was **dropped**.
2. `r3` was **rebooting** when the write happened.
3. `r3` was **restored from a slightly old backup**.
4. A brief **network partition** cut `r3` off for a few seconds.

Any of these leave one replica with the **old** value while the others have the new one. If the coordinator handling a client read just trusts the *first* replica it asks, the client can get a **stale** value — last Tuesday's price, the pre-rename username, an already-deleted post.

This notebook shows the problem concretely so that Notebook 2's solution feels necessary.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/read-repair
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🧱 A tiny replica

Each replica stores `key -> (value, timestamp)`. Higher timestamp = newer write. This is called **last-write-wins (LWW)**: the simplest possible conflict-resolution rule.


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, Optional
import random

Entry = Tuple[str, int]  # (value, timestamp)

@dataclass
class Replica:
    name: str
    data: Dict[str, Entry] = field(default_factory=dict)

    def write(self, k: str, v: str, ts: int) -> None:
        # No LWW check yet — we set whatever we're told. We'll tighten this in NB 2.
        self.data[k] = (v, ts)

    def read(self, k: str) -> Optional[Entry]:
        return self.data.get(k)

# Three replicas holding the user profile for user:42.
# r1 and r2 received the latest write (ts=200, "Alice v2").
# r3 missed that write and still holds the older one (ts=100, "Alice v1").
r1, r2, r3 = Replica("r1"), Replica("r2"), Replica("r3")
r1.write("user:42", "Alice v2", ts=200)
r2.write("user:42", "Alice v2", ts=200)
r3.write("user:42", "Alice v1", ts=100)

for r in (r1, r2, r3):
    print(r.name, "->", r.read("user:42"))


## 🟥 BAD #1: trust the first replica that answers

A naive coordinator picks *one* replica at random (or the closest one) and returns whatever it says. Fast, cheap — and sometimes wrong.


In [ ]:
replicas = [r1, r2, r3]

def naive_read(key: str):
    chosen = random.choice(replicas)
    return chosen.name, chosen.read(key)

random.seed(0)
for _ in range(6):
    print(naive_read("user:42"))


Every time the coin lands on `r3`, the client sees stale `Alice v1`. Let's quantify how bad it is.


## 📈 How often do clients get stale data?

With 1 stale replica out of 3, a uniform-random pick is stale **~33%** of the time. Let's confirm empirically, then see how it gets worse as more replicas drift.


In [ ]:
def stale_rate(num_replicas: int, num_stale: int, trials: int = 20_000, seed: int = 1) -> float:
    fresh_ts, stale_ts = 200, 100
    reps = []
    for i in range(num_replicas):
        r = Replica(f"r{i}")
        ts = stale_ts if i < num_stale else fresh_ts
        val = "Alice v1" if i < num_stale else "Alice v2"
        r.write("user:42", val, ts)
        reps.append(r)
    rng = random.Random(seed)
    stale = 0
    for _ in range(trials):
        # ONE replica is picked and we read from that same one. (Picking the name
        # from one replica and the value from another would still produce the right
        # headline number, but it would not be a read.)
        chosen = rng.choice(reps)
        _, ts = chosen.read("user:42")
        if ts < fresh_ts:
            stale += 1
    return stale / trials

for n, s in [(3, 1), (3, 2), (5, 2), (5, 3)]:
    rate = stale_rate(n, s)
    print(f"{n} replicas, {s} stale : {rate:.1%}")
    # The stale rate for a uniform single-replica read is exactly s/n. Anything else
    # means the sampling is wrong, not that the system is interesting.
    assert abs(rate - s / n) < 0.02, (n, s, rate)
print("\n✔ measured rates match s/n — a read-one coordinator is stale exactly as often")
print("  as the fraction of replicas that are behind")


## 🟥 BAD #2: "read from the closest replica"

In production, coordinators often prefer the *closest* replica (lowest latency). That's fast — but if the closest replica is the one that missed the write, **every** read is stale until something fixes it.


In [ ]:
# Simulate: the app server is in the same datacenter as r3 (the stale one).
def closest_read(key: str):
    return r3.name, r3.read(key)

for _ in range(5):
    print(closest_read("user:42"))   # 100% stale, forever

# This is the case that random sampling hides: not 33% stale, 100% stale, indefinitely.
assert all(closest_read("user:42")[1][0] == "Alice v1" for _ in range(100))
print("\n💥 100/100 reads stale. No retry, no jitter, no eventual consistency will")
print("   fix this on its own — nothing is ever going to write to r3 again.")


## 🎯 Takeaways

- A single-replica read is fast but can return stale data, sometimes **always** stale (the "closest replica is behind" case).
- The real question isn't *if* replicas drift — it's **how quickly we notice and fix it**.

👉 **Notebook 2** introduces **read-repair**: query several replicas on every read, return the freshest value, and push the fresh value back to any replica that was behind. We pay a little extra read latency to heal the cluster as a side effect of normal traffic.
